In [2]:
import pandas as pd
import numpy as np

In [3]:
mhs = pd.read_csv("../data/processed/mhs_main_experiment_annotations.csv")

survey_item_cols = [
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

print(mhs.shape)

(49433, 52)


In [6]:
comment_level = (
    mhs.groupby("comment_id")
    .agg(
        target_type=("target_type", "first"),
        **{f"mean_{col}": (col, "mean") for col in survey_item_cols}
    )
    .reset_index()
)

comment_level.head()

,comment_id,target_type,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,mean_attack_defend,mean_hatespeech
0,6,women_only,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,3.500000,0.000000
1,7,immigrant_only,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,3.500000,1.000000
2,10,immigrant_only,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,3.666667,1.333333
3,11,women_only,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,3.000000,1.500000
4,12,women_only,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,3.000000,1.000000


In [7]:
for col in survey_item_cols:
    mean_col = f"mean_{col}"
    bucket_col = f"{col}_bucket"

    comment_level[bucket_col] = pd.cut(
        comment_level[mean_col],
        bins=[-0.01, 1.33, 2.66, 4.0],
        labels=["low", "medium", "high"],
        include_lowest=True
    )

comment_level.head()

,comment_id,target_type,mean_sentiment,mean_respect,mean_insult,mean_humiliate,mean_status,mean_dehumanize,mean_violence,mean_genocide,...,sentiment_bucket,respect_bucket,insult_bucket,humiliate_bucket,status_bucket,dehumanize_bucket,violence_bucket,genocide_bucket,attack_defend_bucket,hatespeech_bucket
0,6,women_only,3.500000,4.000000,3.500000,4.000000,3.000000,3.500000,2.500000,1.000000,...,high,high,high,high,high,high,medium,low,high,low
1,7,immigrant_only,3.500000,3.500000,1.500000,1.000000,2.500000,1.500000,2.000000,2.000000,...,high,high,medium,low,medium,medium,medium,medium,high,low
2,10,immigrant_only,3.333333,3.666667,3.333333,2.666667,3.333333,2.666667,3.666667,3.333333,...,high,high,high,high,high,high,high,high,high,medium
3,11,women_only,4.000000,4.000000,3.000000,2.500000,2.000000,0.500000,0.500000,0.000000,...,high,high,high,medium,medium,low,low,low,high,medium
4,12,women_only,3.500000,4.000000,3.500000,3.000000,2.500000,1.500000,0.500000,0.000000,...,high,high,high,high,medium,medium,low,low,high,low


In [8]:
bucket_cols = [f"{col}_bucket" for col in survey_item_cols]

comment_level["all_survey_strata"] = (
    comment_level["target_type"].astype(str)
    + "__"
    + comment_level[bucket_cols].astype(str).agg("_".join, axis=1)
)

strata_counts = comment_level["all_survey_strata"].value_counts()

print("Unique comments:", comment_level["comment_id"].nunique())
print("Unique strata:", strata_counts.shape[0])
print("Smallest stratum size:", strata_counts.min())
print("Largest stratum size:", strata_counts.max())

strata_counts.describe()

Unique comments: 19761
Unique strata: 2360
Smallest stratum size: 1
Largest stratum size: 669


count    2360.000000
mean        8.373305
std        33.734244
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max       669.000000
Name: count, dtype: float64

In [9]:
too_small = strata_counts[strata_counts < 5]

print("Strata with fewer than 5 comments:", len(too_small))
print("Percentage of strata too small:", round(len(too_small) / len(strata_counts) * 100, 2))

too_small.head(20)

Strata with fewer than 5 comments: 1874
Percentage of strata too small: 79.41


all_survey_strata
immigrant_only__high_high_medium_medium_high_medium_medium_low_medium_low    4
immigrant_only__high_high_high_medium_medium_high_low_low_high_medium        4
immigrant_only__high_high_high_high_medium_high_medium_medium_high_medium    4
immigrant_only__high_high_high_high_low_low_low_low_high_low                 4
immigrant_only__medium_high_medium_low_medium_low_low_low_medium_low         4
immigrant_only__high_high_high_medium_high_high_medium_low_high_medium       4
women_only__medium_medium_medium_medium_low_low_low_low_low_low              4
immigrant_only__medium_low_low_medium_medium_low_low_low_low_low             4
women_and_immigrant__high_high_high_medium_high_low_low_low_high_low         4
women_only__low_medium_low_low_high_low_low_low_medium_low                   4
women_only__high_medium_medium_medium_high_low_low_low_high_low              4
immigrant_only__high_high_medium_medium_medium_low_medium_medium_high_low    4
women_only__medium_high_medium_hig

In [10]:
mean_cols = [f"mean_{col}" for col in survey_item_cols]

comment_level["composite_severity"] = comment_level[mean_cols].mean(axis=1)

comment_level["composite_bucket"] = pd.qcut(
    comment_level["composite_severity"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

comment_level["composite_strata"] = (
    comment_level["target_type"].astype(str)
    + "__"
    + comment_level["composite_bucket"].astype(str)
)

composite_counts = comment_level["composite_strata"].value_counts().sort_index()

composite_counts

composite_strata
immigrant_only__high           2229
immigrant_only__low            3155
immigrant_only__medium         2619
women_and_immigrant__high       137
women_and_immigrant__low        150
women_and_immigrant__medium      97
women_only__high               3758
women_only__low                3634
women_only__medium             3982
Name: count, dtype: int64

In [11]:
comment_level["hatespeech_bucket"] = pd.cut(
    comment_level["mean_hatespeech"],
    bins=[-0.01, 0.5, 1.5, 2.0],
    labels=["low", "medium", "high"],
    include_lowest=True
)

comment_level["hatespeech_strata"] = (
    comment_level["target_type"].astype(str)
    + "__"
    + comment_level["hatespeech_bucket"].astype(str)
)

hatespeech_counts = comment_level["hatespeech_strata"].value_counts().sort_index()

hatespeech_counts

hatespeech_strata
immigrant_only__high           1092
immigrant_only__low            5526
immigrant_only__medium         1385
women_and_immigrant__high        82
women_and_immigrant__low        231
women_and_immigrant__medium      71
women_only__high               2359
women_only__low                6476
women_only__medium             2539
Name: count, dtype: int64

In [13]:
strata_counts.describe()


count    2360.000000
mean        8.373305
std        33.734244
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max       669.000000
Name: count, dtype: float64

In [14]:
len(too_small)

1874

In [15]:
composite_counts

composite_strata
immigrant_only__high           2229
immigrant_only__low            3155
immigrant_only__medium         2619
women_and_immigrant__high       137
women_and_immigrant__low        150
women_and_immigrant__medium      97
women_only__high               3758
women_only__low                3634
women_only__medium             3982
Name: count, dtype: int64

In [16]:
hatespeech_counts

hatespeech_strata
immigrant_only__high           1092
immigrant_only__low            5526
immigrant_only__medium         1385
women_and_immigrant__high        82
women_and_immigrant__low        231
women_and_immigrant__medium      71
women_only__high               2359
women_only__low                6476
women_only__medium             2539
Name: count, dtype: int64